# CIS 5270 — DPO Extra Runs

This notebook runs **three additional DPO experiments in parallel** with
different hyperparameter and corruption-strategy configurations.

The first DPO run (the one currently training from Friday afternoon) uses
the full 10k-example dataset with batch size 1 and the default mixed
corruption strategy. That single run is so slow because every gradient step
processes one example. The three runs in this notebook are designed to
finish in **2–4 hours each** by dramatically reducing the step count via
smaller datasets and larger batches, so we can have multiple DPO models to
compare in the report.

| Run | Dataset | Batch | LR mult | Corruption mix | Steps |
|-----|---------|-------|---------|----------------|-------|
| A (already running, separate notebook) | 10k | 1 | 1.0 | default mix | ~10000 |
| **B (this notebook)** | 2.5k | 4 | 1.0 | default mix | ~625 |
| **C (this notebook)** | 2.5k | 4 | 1.0 | trace-error only | ~625 |
| **D (this notebook)** | 2.5k | 4 | 2.0 | default mix | ~625 |

**Investigated questions:**
- **B vs A:** does data scale matter, or is 2.5k examples enough?
- **C vs B:** which corruption strategy carries the signal — diverse error types, or the targeted "model contradicts its own reasoning trace" failure mode?
- **D vs B:** is DPO sensitive to learning rate?

All three jobs run **in parallel** on Azure's GlobalStandard pool.


## Section 0 — Imports, Generators, Data-Gen Utilities

These cells are copied verbatim from the main notebook (Sections 0.1–0.4,
0.5b). They define `VARIABLES`, `SYSTEM_PROMPT`, `generate_unique`,
`generate_no_solution`, `generate_infinite`, `verify_solution_type`,
`system_to_string`, `generate_reasoning_trace`, `make_sft_record`,
`make_eval_record`, the corruption helpers, and `make_dpo_record` /
`build_dpo_split`.

Run all cells in this section before moving on.


In [1]:
import numpy as np
import random
import json
import re
import os
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Reproducibility ──────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# ── Variable pool (always alphabetical — drives output format) ────────────────
VARIABLES = ['x', 'y', 'z', 'w', 'u', 'v']

# ── System size ──────────────────────────────────────────────────────────────────────────
N_VARS_MIN = 2
N_VARS_MAX = 3

# ── Coefficient ranges  (np.random.randint convention: [low, high)) ───────────────
COEFF_RANGE  = (-6, 7)    # A-matrix integer entries:        -6 … +6
X_STAR_RANGE = (-5, 6)    # Ground-truth solution entries:   -5 … +5
LC_RANGE     = (-3, 4)    # Linear-combination weights:      -3 … +3

# ── Dataset sizes ───────────────────────────────────────────────────────────────────────
N_TRAIN_TOTAL = 10000
N_VAL_TOTAL   =  1000
N_EVAL_TOTAL  =  1000

TYPE_WEIGHTS   = {'unique': 2.0, 'no_solution': 0.5, 'infinite': 0.5}
N_VARS_WEIGHTS = {2: 1.0, 3: 2.0}

# ── LLM inference tunables ────────────────────────────────────────────────────────────
MAX_TOKENS           = 200   # max tokens for direct-answer responses
REASONING_MAX_TOKENS = 512   # max tokens for reasoning-trace responses
TEMPERATURE          = 0.0
REQUEST_TIMEOUT      = 20

# ── Output paths ──────────────────────────────────────────────────────────────────────────────
TRAIN_JSONL = 'training.jsonl'
VAL_JSONL   = 'validation.jsonl'
EVAL_JSONL  = 'eval.jsonl'

# ── Reasoning-trace toggle ────────────────────────────────────────────────────────────
# Set to True to include step-by-step Gauss–Jordan traces in every assistant
# turn and 'answer' field.  All three datasets (train, val, eval) flip together.
# The parser and eval harness adjust automatically based on this flag.
INCLUDE_REASONING = True

# ── System prompts ───────────────────────────────────────────────────────────────────────
# SYSTEM_PROMPT is set automatically from the flag; do not edit it directly.

SYSTEM_PROMPT_DIRECT = (
    "You solve systems of linear equations with integer coefficients. "
    "Output EXACTLY ONE of the following three formats and nothing else. "
    "Do NOT explain your reasoning or add any other text.\n"
    "(1) Unique solution — list every variable assignment comma-separated "
    "in alphabetical order, e.g.: x=2, y=-1, z=0\n"
    "(2) No solution — output exactly: NO SOLUTION\n"
    "(3) Infinitely many solutions — output exactly: INFINITE"
)

SYSTEM_PROMPT_REASONING = (
    "You solve systems of linear equations with integer coefficients. "
    "Place your final answer on its own line starting with \"ANSWER:\" "
    "using EXACTLY ONE of the following three formats:\n"
    "(1) Unique solution — list every variable assignment comma-separated "
    "in alphabetical order: ANSWER: x=2, y=-1, z=0\n"
    "(2) No solution: ANSWER: NO SOLUTION\n"
    "(3) Infinitely many solutions: ANSWER: INFINITE"
)

SYSTEM_PROMPT = SYSTEM_PROMPT_REASONING if INCLUDE_REASONING else SYSTEM_PROMPT_DIRECT

# ── Numerical tolerances ──────────────────────────────────────────────────────────────────
_RANK_TOL     = 1e-9
_RESIDUAL_TOL = 1e-6

# ── Generator retry limits ───────────────────────────────────────────────────────────────
_MAX_ROW_RETRIES    = 300
_MAX_LC_RETRIES     = 100
_MAX_UNIQUE_RETRIES = 500
_MAX_NO_SOL_RETRIES = 300
_MAX_PERTURB_TRIES  = 100
_PERTURBATION_DELTAS = [-4, -3, -2, -1, 1, 2, 3, 4]

# ── Distribution spot-check ───────────────────────────────────────────────────────────────
N_SAMPLES = 2


print('Global configuration loaded.')
print(f'  n_vars range     : {N_VARS_MIN} – {N_VARS_MAX}')
print(f'  COEFF_RANGE      : {COEFF_RANGE}')
print(f'  X_STAR_RANGE     : {X_STAR_RANGE}')
print(f'  LC_RANGE         : {LC_RANGE}')
print(f'  Train/Val/Eval   : {N_TRAIN_TOTAL} / {N_VAL_TOTAL} / {N_EVAL_TOTAL}')
print(f'  TYPE_WEIGHTS     : {TYPE_WEIGHTS}')
print(f'  N_VARS_WEIGHTS   : {N_VARS_WEIGHTS}')
print(f'  Tolerances       : rank={_RANK_TOL}, residual={_RESIDUAL_TOL}')
print(f'  MAX_TOKENS       : {MAX_TOKENS} (direct) / {REASONING_MAX_TOKENS} (reasoning)')
print(f'  INCLUDE_REASONING: {INCLUDE_REASONING}')
print(f'  Active prompt    : {"SYSTEM_PROMPT_REASONING" if INCLUDE_REASONING else "SYSTEM_PROMPT_DIRECT"}')


Global configuration loaded.
  n_vars range     : 2 – 3
  COEFF_RANGE      : (-6, 7)
  X_STAR_RANGE     : (-5, 6)
  LC_RANGE         : (-3, 4)
  Train/Val/Eval   : 10000 / 1000 / 1000
  TYPE_WEIGHTS     : {'unique': 2.0, 'no_solution': 0.5, 'infinite': 0.5}
  N_VARS_WEIGHTS   : {2: 1.0, 3: 2.0}
  Tolerances       : rank=1e-09, residual=1e-06
  MAX_TOKENS       : 200 (direct) / 512 (reasoning)
  INCLUDE_REASONING: True
  Active prompt    : SYSTEM_PROMPT_REASONING


In [2]:
def _make_rank_deficient(n_vars):
    """
    Build an n_vars × n_vars integer matrix with rank < n_vars.

    Between 1 and n_vars-1 rows are random integer linear combinations of
    the remaining n_independent = n_vars - n_dep base rows.

    Returns
    -------
    A     : np.ndarray[int], shape (n_vars, n_vars)
    n_dep : int  —  number of dependent rows inserted
    """
    n_dep = random.randint(1, n_vars - 1)
    n_ind = n_vars - n_dep

    # Draw n_ind genuinely linearly-independent rows
    for _ in range(_MAX_ROW_RETRIES):
        base = np.random.randint(*COEFF_RANGE, size=(n_ind, n_vars))
        if np.linalg.matrix_rank(base.astype(float), tol=_RANK_TOL) == n_ind:
            break
    else:
        raise RuntimeError(
            f'_make_rank_deficient: could not find {n_ind} independent rows '
            f'in {_MAX_ROW_RETRIES} attempts (n_vars={n_vars}, COEFF_RANGE={COEFF_RANGE})'
        )

    # Build each dependent row as a non-trivial LC of the base rows
    dep_rows = []
    for _ in range(n_dep):
        for _ in range(_MAX_LC_RETRIES):
            lc = np.random.randint(*LC_RANGE, size=n_ind)
            if np.any(lc != 0):
                dep_rows.append(lc @ base)
                break
        else:
            raise RuntimeError(
                f'_make_rank_deficient: LC_RANGE={LC_RANGE} produced only '
                f'all-zero coefficients after {_MAX_LC_RETRIES} attempts'
            )

    # Interleave independent and dependent rows randomly
    all_rows = list(base) + [np.asarray(r, dtype=int) for r in dep_rows]
    random.shuffle(all_rows)
    return np.array(all_rows, dtype=int), n_dep


# ─────────────────────────────────────────────────────────────────────────────

def generate_unique(n_vars):
    """
    Full-rank n_vars × n_vars system with a unique integer solution.

    Construction:
      1. Sample an invertible integer matrix A (rank == n_vars).
      2. Sample integer x*, then set b = A @ x*.

    Returns: A, b, x_star  (all np.ndarray, dtype int)
    """
    for _ in range(_MAX_UNIQUE_RETRIES):
        A = np.random.randint(*COEFF_RANGE, size=(n_vars, n_vars))
        if np.linalg.matrix_rank(A.astype(float), tol=_RANK_TOL) == n_vars:
            break
    else:
        raise RuntimeError(
            f'generate_unique: could not find invertible A in {_MAX_UNIQUE_RETRIES} attempts '
            f'(n_vars={n_vars}, COEFF_RANGE={COEFF_RANGE})'
        )
    x_star = np.random.randint(*X_STAR_RANGE, size=n_vars)
    b = A @ x_star
    return A, b, x_star


def generate_infinite(n_vars):
    """
    Rank-deficient n_vars × n_vars system with infinitely many solutions.

    Construction:
      1. Build rank-deficient A via _make_rank_deficient.
      2. Sample x*, compute b = A @ x*.
         => rank(A) = rank([A|b]) < n_vars  (consistent, underdetermined).

    Returns: A, b (np.ndarray, int), n_dep (int)
    """
    A, n_dep = _make_rank_deficient(n_vars)
    x_star   = np.random.randint(*X_STAR_RANGE, size=n_vars)
    b        = A @ x_star
    return A, b, n_dep


def generate_no_solution(n_vars):
    """
    Rank-deficient n_vars × n_vars system with no solution.

    Construction:
      1. Build rank-deficient A (identical structure to generate_infinite).
      2. Start from a consistent b = A @ x_ref.
      3. Perturb one row entry of b by a nonzero integer delta until
         rank([A|b]) > rank(A)  (inconsistent).

    Why single-entry perturbations work: for rank-deficient integer A,
    the column space col(A) is a proper subspace of Z^n.  A random
    integer delta*e_i is almost never in col(A), so a small number of
    (row, delta) pairs is sufficient in practice.  The outer retry loop
    regenerates A if an unusually degenerate case stalls.

    Retry limits: _MAX_NO_SOL_RETRIES outer, _MAX_PERTURB_TRIES inner.
    Perturbation deltas: _PERTURBATION_DELTAS (all defined in Section 0.1).

    Returns: A, b (np.ndarray, int), n_dep (int)
    """
    for _outer in range(_MAX_NO_SOL_RETRIES):
        A, n_dep = _make_rank_deficient(n_vars)
        rank_A   = np.linalg.matrix_rank(A.astype(float), tol=_RANK_TOL)

        x_ref  = np.random.randint(*X_STAR_RANGE, size=n_vars)
        b_base = (A @ x_ref).copy()

        for _inner in range(_MAX_PERTURB_TRIES):
            row   = random.randint(0, n_vars - 1)
            delta = random.choice(_PERTURBATION_DELTAS)
            b_test = b_base.copy()
            b_test[row] += delta
            A_aug = np.column_stack([A.astype(float),
                                     b_test.reshape(-1, 1).astype(float)])
            if np.linalg.matrix_rank(A_aug, tol=_RANK_TOL) > rank_A:
                return A, b_test, n_dep

    raise RuntimeError(
        f'generate_no_solution: exhausted {_MAX_NO_SOL_RETRIES} outer retries for n_vars={n_vars}'
    )


# ─────────────────────────────────────────────────────────────────────────────

def verify_solution_type(A, b):
    """
    Derive ground-truth solution type from Rouché-Capelli rank conditions.
    Used as a sanity-check assertion during generation.

    rank(A) == rank([A|b]) == n  =>  unique
    rank(A) == rank([A|b])  < n  =>  infinite
    rank(A)  < rank([A|b])       =>  no_solution
    """
    n    = A.shape[1]
    r_A  = np.linalg.matrix_rank(A.astype(float), tol=_RANK_TOL)
    r_Ab = np.linalg.matrix_rank(
        np.column_stack([A.astype(float), b.reshape(-1, 1).astype(float)]),
        tol=_RANK_TOL
    )
    if   r_A == r_Ab == n: return 'unique'
    elif r_A == r_Ab  < n: return 'infinite'
    else:                  return 'no_solution'


print('Generators loaded: generate_unique, generate_infinite, generate_no_solution')
print('Verifier loaded:   verify_solution_type')


Generators loaded: generate_unique, generate_infinite, generate_no_solution
Verifier loaded:   verify_solution_type


In [3]:
def system_to_string(A, b, n_vars):
    """
    Render the n_vars × n_vars system (A, b) as a multi-line string.

    Parameters
    ----------
    A      : array-like (int), shape (n_vars, n_vars)
    b      : array-like (int), shape (n_vars,)
    n_vars : int

    Returns
    -------
    str  — one equation per line, e.g. '2x - 3y = 5\n-x + z = 0'
    """
    var_names = VARIABLES[:n_vars]
    lines = []
    for i in range(len(A)):
        terms = []
        for j in range(n_vars):
            c = int(A[i][j])
            if c == 0:
                continue
            v = var_names[j]
            if not terms:           # first nonzero term in this row
                if   c ==  1: terms.append(v)
                elif c == -1: terms.append(f'-{v}')
                else:         terms.append(f'{c}{v}')
            else:                   # subsequent terms
                if   c ==  1: terms.append(f'+ {v}')
                elif c == -1: terms.append(f'- {v}')
                elif c  >  0: terms.append(f'+ {c}{v}')
                else:         terms.append(f'- {abs(c)}{v}')
        lhs = ' '.join(terms) if terms else '0'
        lines.append(f'{lhs} = {int(b[i])}')
    return '\n'.join(lines)


# Quick sanity check
_A = np.array([[2, -3, 1], [0, 4, -2], [1, 0, 3]])
_b = np.array([5, -2, 7])
print('system_to_string test:')
print(system_to_string(_A, _b, 3))

system_to_string test:
2x - 3y + z = 5
4y - 2z = -2
x + 3z = 7


In [4]:
from math import gcd
from functools import reduce


def _reduce_row(row):
    # GCD-reduce and sign-normalise so the leading nonzero entry is positive.
    vals = [abs(v) for v in row if v != 0]
    if not vals:
        return list(row)
    g = reduce(gcd, vals)
    reduced = [v // g for v in row]
    for v in reduced:
        if v != 0:
            if v < 0:
                reduced = [-x for x in reduced]
            break
    return reduced


def _aug_row_to_eq(row, var_names):
    # Format augmented-matrix row [c0, ..., c_{n-1}, rhs] as an equation string.
    n = len(var_names)
    terms = []
    for j in range(n):
        c = row[j]
        if c == 0:
            continue
        v = var_names[j]
        if not terms:
            if   c ==  1: terms.append(v)
            elif c == -1: terms.append(f'-{v}')
            else:         terms.append(f'{c}{v}')
        else:
            if   c ==  1: terms.append(f'+ {v}')
            elif c == -1: terms.append(f'- {v}')
            elif c  >  0: terms.append(f'+ {c}{v}')
            else:         terms.append(f'- {abs(c)}{v}')
    lhs = ' '.join(terms) if terms else '0'
    return f'{lhs} = {row[n]}'


def generate_reasoning_trace(A, b, n_vars, solution_type):
    # Produce a step-by-step Gauss-Jordan elimination trace for Ax = b.
    # Uses integer-only arithmetic (LCM-based row ops + GCD reduction).
    # No fractions appear at any step.  The returned string ends with
    # a line of the form 'ANSWER: ...' that the correctness parser extracts.
    var_names = VARIABLES[:n_vars]
    aug = [[int(A[i][j]) for j in range(n_vars)] + [int(b[i])] for i in range(n_vars)]

    lines = []
    for row in aug:
        lines.append(_aug_row_to_eq(row, var_names))
    lines.append('')

    pivot_row = 0
    for col in range(n_vars):
        # Find first nonzero pivot at or below current pivot_row
        pivot = next((r for r in range(pivot_row, n_vars) if aug[r][col] != 0), None)
        if pivot is None:
            continue   # free-variable column

        if pivot != pivot_row:
            aug[pivot_row], aug[pivot] = aug[pivot], aug[pivot_row]
            lines.append(f'Swap R{pivot_row+1} <-> R{pivot+1}')

        # Normalise pivot row: GCD-reduce + positive leading coefficient
        old_pivot = list(aug[pivot_row])
        aug[pivot_row] = _reduce_row(aug[pivot_row])
        p = aug[pivot_row][col]   # always positive after normalisation
        if aug[pivot_row] != old_pivot:
            old_vals = [abs(v) for v in old_pivot if v != 0]
            g = reduce(gcd, old_vals) if old_vals else 1
            sign_flip = old_pivot[next(i for i, v in enumerate(old_pivot) if v != 0)] < 0
            if sign_flip and g > 1:
                norm_op = f'R{pivot_row+1} -> -R{pivot_row+1} / {g}'
            elif sign_flip:
                norm_op = f'R{pivot_row+1} -> -R{pivot_row+1}'
            else:
                norm_op = f'R{pivot_row+1} -> R{pivot_row+1} / {g}'
            lines.append(f'{norm_op}: {_aug_row_to_eq(aug[pivot_row], var_names)}')

        # Gauss-Jordan: eliminate this column from ALL other rows
        for r in range(n_vars):
            if r == pivot_row or aug[r][col] == 0:
                continue
            a = aug[r][col]

            if a % p == 0:
                # Integer-multiplier form: R_r -> R_r - m*R_pivot
                m = a // p
                new_row = [aug[r][j] - m * aug[pivot_row][j] for j in range(n_vars + 1)]
                if   m ==  1: op = f'R{r+1} -> R{r+1} - R{pivot_row+1}'
                elif m == -1: op = f'R{r+1} -> R{r+1} + R{pivot_row+1}'
                elif m  >  0: op = f'R{r+1} -> R{r+1} - {m}*R{pivot_row+1}'
                else:         op = f'R{r+1} -> R{r+1} + {abs(m)}*R{pivot_row+1}'
            else:
                # LCM-scale form: R_r -> p*R_r - a*R_pivot (always integer)
                new_row = [p * aug[r][j] - a * aug[pivot_row][j] for j in range(n_vars + 1)]
                op = (f'R{r+1} -> {p}*R{r+1} - {a}*R{pivot_row+1}'
                      if a > 0 else
                      f'R{r+1} -> {p}*R{r+1} + {abs(a)}*R{pivot_row+1}')

            new_row = _reduce_row(new_row)
            aug[r] = new_row
            eq_str = _aug_row_to_eq(aug[r], var_names)

            # Detect contradiction or free-variable row immediately
            if all(aug[r][c] == 0 for c in range(n_vars)):
                if aug[r][n_vars] != 0:
                    lines += [f'{op}: {eq_str}',
                               'Contradiction - no solution exists', '',
                               'ANSWER: NO SOLUTION']
                else:
                    lines += [f'{op}: 0 = 0 (free variable - infinitely many solutions)', '',
                               'ANSWER: INFINITE']
                return '\n'.join(lines)

            lines.append(f'{op}: {eq_str}')

        pivot_row += 1

    # Fully reduced: read off unique solution
    solution = {}
    for r in range(n_vars):
        pivot_col = next((c for c in range(n_vars) if aug[r][c] != 0), None)
        if pivot_col is None:
            continue
        coeff = aug[r][pivot_col]
        val   = aug[r][n_vars] // coeff
        solution[var_names[pivot_col]] = val

    lines.append('')
    answer_str = ', '.join(f'{v}={solution[v]}' for v in var_names)
    lines.append(f'ANSWER: {answer_str}')
    return '\n'.join(lines)


# -- Sanity checks ------------------------------------------------------------
_A2u = np.array([[6, -3], [-6, -4]])
_b2u = np.array([3, 32])
print('2-var unique (expect x=-2, y=-5):')
print(generate_reasoning_trace(_A2u, _b2u, 2, 'unique'))
print()

_Ainf = np.array([[2, -5], [-6, 15]])
_binf = np.array([-1, 3])
print('2-var infinite:')
print(generate_reasoning_trace(_Ainf, _binf, 2, 'infinite'))
print()

_Ans = np.array([[2, -5], [-6, 15]])
_bns = np.array([-1, 4])
print('2-var no_solution:')
print(generate_reasoning_trace(_Ans, _bns, 2, 'no_solution'))


2-var unique (expect x=-2, y=-5):
6x - 3y = 3
-6x - 4y = 32

R1 -> R1 / 3: 2x - y = 1
R2 -> R2 + 3*R1: y = -5
R1 -> R1 + R2: x = -2

ANSWER: x=-2, y=-5

2-var infinite:
2x - 5y = -1
-6x + 15y = 3

R2 -> R2 + 3*R1: 0 = 0 (free variable - infinitely many solutions)

ANSWER: INFINITE

2-var no_solution:
2x - 5y = -1
-6x + 15y = 4

R2 -> R2 + 3*R1: 0 = 1
Contradiction - no solution exists

ANSWER: NO SOLUTION


In [5]:
def _format_answer(solution_type, ground_truth, n_vars):
    # Canonical string representation of the correct answer.
    if solution_type == 'unique':
        return ', '.join(f'{v}={ground_truth[v]}' for v in VARIABLES[:n_vars])
    elif solution_type == 'no_solution':
        return 'NO SOLUTION'
    else:
        return 'INFINITE'


def _assistant_content(A, b, n_vars, solution_type, answer_str):
    # Return the assistant turn: reasoning trace (if enabled) or direct answer.
    # A and b may be None when INCLUDE_REASONING is False (backward compat).
    if INCLUDE_REASONING and A is not None and b is not None:
        return generate_reasoning_trace(A, b, n_vars, solution_type)
    return answer_str


def make_sft_record(system_text, solution_type, ground_truth, n_vars, A=None, b=None):
    # Build one JSONL record for Supervised Fine-Tuning.
    #
    # Azure Foundry SFT format:
    #   { 'messages': [
    #       {'role': 'system',    'content': '...'},
    #       {'role': 'user',      'content': '...'},
    #       {'role': 'assistant', 'content': '...'}   <- gold label
    #   ]}
    answer_str = _format_answer(solution_type, ground_truth, n_vars)
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': system_text},
            {'role': 'assistant', 'content': _assistant_content(A, b, n_vars,
                                                                 solution_type,
                                                                 answer_str)},
        ]
    }


def make_eval_record(system_text, solution_type, ground_truth, A, b, n_vars, n_dep):
    # Build one record for the offline evaluation harness.
    #
    # The 'answer' field mirrors the assistant format controlled by INCLUDE_REASONING
    # so all datasets stay consistent.  compute_reward uses A/b directly and does
    # not depend on the answer string format.
    answer_str = _format_answer(solution_type, ground_truth, n_vars)
    answer = _assistant_content(A, b, n_vars, solution_type, answer_str)
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': system_text},
        ],
        'answer':        answer,
        'solution_type': solution_type,
        'ground_truth':  ground_truth,
        'n_vars':        n_vars,
        'n_dependent':   n_dep,
        'A':             [list(map(int, row)) for row in A],
        'b':             list(map(int, b)),
    }


print('Record builders loaded: make_sft_record, make_eval_record')


Record builders loaded: make_sft_record, make_eval_record


In [6]:
def _generate_split(n_total, label):
    """
    Generate a split of n_total examples, distributed across solution types
    and n_vars values according to TYPE_WEIGHTS and N_VARS_WEIGHTS.

    Returns
    -------
    sft_records  : list[dict]   — for training.jsonl / validation.jsonl
    eval_records : list[dict]   — for eval.jsonl
    meta         : list[dict]   — statistics for visualisation
    """
    sft_records, eval_records, meta = [], [], []

    # Compute per-type counts that sum exactly to n_total
    sol_types = ['unique', 'no_solution', 'infinite']
    _total_w  = sum(TYPE_WEIGHTS[t] for t in sol_types)
    _exact    = {t: n_total * TYPE_WEIGHTS[t] / _total_w for t in sol_types}
    _counts   = {t: int(_exact[t]) for t in sol_types}
    _rem      = n_total - sum(_counts.values())
    # Distribute rounding remainder to types with largest fractional parts
    for t in sorted(sol_types, key=lambda t: _exact[t] - _counts[t], reverse=True)[:_rem]:
        _counts[t] += 1

    _nvars_keys    = list(N_VARS_WEIGHTS.keys())
    _nvars_weights = list(N_VARS_WEIGHTS.values())

    for sol_type in sol_types:
        n_this = _counts[sol_type]
        print(f'  [{label}] {sol_type:<12} ... ', end='', flush=True)
        for _ in range(n_this):
            n = random.choices(_nvars_keys, weights=_nvars_weights, k=1)[0]

            if sol_type == 'unique':
                A, b, x_star = generate_unique(n)
                gt    = {VARIABLES[j]: int(x_star[j]) for j in range(n)}
                n_dep = 0
            elif sol_type == 'no_solution':
                A, b, n_dep = generate_no_solution(n)
                gt = None
            else:
                A, b, n_dep = generate_infinite(n)
                gt = None

            # Sanity check — will raise AssertionError if generator is broken
            true_type = verify_solution_type(A, b)
            assert true_type == sol_type, (
                f'Verification failed: requested {sol_type!r}, got {true_type!r}\n'
                f'A={A.tolist()}, b={b.tolist()}'
            )

            sys_text = system_to_string(A, b, n)
            sft_records.append(make_sft_record(sys_text, sol_type, gt, n, A=A, b=b))
            eval_records.append(make_eval_record(sys_text, sol_type, gt, A, b, n, n_dep))
            meta.append({
                'solution_type': sol_type,
                'n_vars':        n,
                'n_dependent':   n_dep,
                'coefficients':  [int(c) for c in A.flatten()],
                'b_values':      [int(v) for v in b],
                'x_star': [int(x_star[j]) for j in range(n)] if sol_type == 'unique' else None,
            })
        print(f'{n_this} done')

    # Shuffle while keeping sft / eval / meta in sync
    idx = list(range(len(sft_records)))
    random.shuffle(idx)
    return ([sft_records[i]  for i in idx],
            [eval_records[i] for i in idx],
            [meta[i]         for i in idx])


def write_jsonl(records, path):
    """Write JSONL with UTF-8 BOM (required by Azure Foundry)."""
    with open(path, 'w', encoding='utf-8-sig') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
    kb = os.path.getsize(path) / 1024
    print(f'  wrote {len(records):>5} records  ->  {path}  ({kb:.1f} KB)')


# ── Main generation ───────────────────────────────────────────────────────────
print('Generating training split ...')
train_sft, train_eval, train_meta = _generate_split(N_TRAIN_TOTAL, 'TRAIN')

print('\nGenerating validation split ...')
val_sft, val_eval, val_meta = _generate_split(N_VAL_TOTAL, 'VAL')

print('\nGenerating held-out evaluation split ...')
# held_out_eval: eval-format records (with A, b, answer, metadata).
# Written to EVAL_JSONL on disk; also kept in memory for immediate use in
# Sections 0.6, 0.8, 2.5.1, 7, and 7.5.
_, held_out_eval, eval_meta = _generate_split(N_EVAL_TOTAL, 'EVAL')

print('\nWriting JSONL files ...')
write_jsonl(train_sft,      TRAIN_JSONL)
write_jsonl(val_sft,        VAL_JSONL)
write_jsonl(held_out_eval,  EVAL_JSONL)

print('\n✓  training.jsonl and validation.jsonl are ready for Section 3.')


Generating training split ...
  [TRAIN] unique       ... 6667 done
  [TRAIN] no_solution  ... 1667 done
  [TRAIN] infinite     ... 1666 done

Generating validation split ...
  [VAL] unique       ... 666 done
  [VAL] no_solution  ... 167 done
  [VAL] infinite     ... 167 done

Generating held-out evaluation split ...
  [EVAL] unique       ... 666 done
  [EVAL] no_solution  ... 167 done
  [EVAL] infinite     ... 167 done

Writing JSONL files ...
  wrote 10000 records  ->  training.jsonl  (7347.6 KB)
  wrote  1000 records  ->  validation.jsonl  (738.7 KB)
  wrote  1000 records  ->  eval.jsonl  (859.3 KB)

✓  training.jsonl and validation.jsonl are ready for Section 3.


In [7]:
# ---------------------------------------------------------------------------
# Parsing
# ---------------------------------------------------------------------------

def _parse_model_output_impl(text, n_vars):
    # Core parser: operates on text that has already been extracted by
    # _extract_for_parsing (ANSWER: delimiter stripped when INCLUDE_REASONING).
    # Tries the last non-empty line first, then falls back to the full response.
    if not text or not text.strip():
        return 'unparseable', None

    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    candidates = []
    if lines:
        candidates.append(lines[-1])
    candidates.append(text.strip())

    no_sol_pat = re.compile(
        r'no[\s_-]?solutions?(?:\s+exists?)?|inconsistent',
        re.IGNORECASE
    )
    infinite_pat = re.compile(
        r'\binfinite(?:ly)?(?:\s+many)?(?:\s+solutions?)?\b',
        re.IGNORECASE
    )

    for cand in candidates:
        cand_stripped = cand.rstrip('.,;:!? ')
        if re.search(no_sol_pat, cand_stripped):
            return 'no_solution', None
        if re.search(infinite_pat, cand_stripped):
            return 'infinite', None

    var_names = VARIABLES[:n_vars]
    for cand in candidates:
        assignments = {}
        for v in var_names:
            m = re.search(v + r'\s*=\s*(-?\d+)', cand, re.IGNORECASE)
            if m:
                assignments[v] = int(m.group(1))
        if len(assignments) == n_vars:
            return 'unique', assignments

    return 'unparseable', None


def _extract_for_parsing(text):
    # When INCLUDE_REASONING is on, extract the content after the ANSWER: delimiter.
    # Falls back to the full text if no delimiter is found (backward compatible).
    if INCLUDE_REASONING:
        m = re.search(r'(?:^|\n)ANSWER:\s*(.+)', text, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return text


def parse_model_output(text, n_vars):
    # Parse raw model output into (solution_type, assignments).
    # Automatically handles both direct-answer and reasoning-trace formats
    # depending on INCLUDE_REASONING.
    return _parse_model_output_impl(_extract_for_parsing(text), n_vars)


# ---------------------------------------------------------------------------
# Reward
# ---------------------------------------------------------------------------

def compute_reward(pred_type, pred_vals, true_type, A, b, n_vars):
    # Binary +/-1.0 reward.
    # Unique solutions verified by substitution (residual < _RESIDUAL_TOL).
    if pred_type != true_type:
        return -1.0

    if true_type == 'unique':
        if pred_vals is None:
            return -1.0
        A_f   = np.array(A, dtype=float)
        b_f   = np.array(b, dtype=float)
        x_hat = np.array([pred_vals.get(v, 0) for v in VARIABLES[:n_vars]], dtype=float)
        residual = float(np.max(np.abs(A_f @ x_hat - b_f)))
        return 1.0 if residual < _RESIDUAL_TOL else -1.0

    return 1.0   # no_solution or infinite: type match is sufficient


def evaluate_batch(eval_records, model_outputs):
    # Convenience wrapper: score a list of (eval_record, model_output_text) pairs.
    results = []
    for rec, output in zip(eval_records, model_outputs):
        n     = rec['n_vars']
        ptype, pvals = parse_model_output(output, n)
        rew   = compute_reward(ptype, pvals, rec['solution_type'],
                               rec['A'], rec['b'], n)
        results.append({
            'solution_type': rec['solution_type'],
            'n_vars':        n,
            'pred_type':     ptype,
            'reward':        rew,
            'raw_output':    output,
        })
    return results


print('Grading functions loaded.')
print('  parse_model_output(text, n_vars)  ->  (solution_type, assignments | None)')
print('  compute_reward(pred_type, pred_vals, true_type, A, b, n_vars)  ->  +/-1.0')
print('  evaluate_batch(eval_records, model_outputs)  ->  list[result_dict]')
print(f'  INCLUDE_REASONING={INCLUDE_REASONING}: parser uses',
      'ANSWER: delimiter extraction' if INCLUDE_REASONING else 'direct parsing (no delimiter)')


Grading functions loaded.
  parse_model_output(text, n_vars)  ->  (solution_type, assignments | None)
  compute_reward(pred_type, pred_vals, true_type, A, b, n_vars)  ->  +/-1.0
  evaluate_batch(eval_records, model_outputs)  ->  list[result_dict]
  INCLUDE_REASONING=True: parser uses ANSWER: delimiter extraction


In [8]:
# ── DPO preference-pair generation ────────────────────────────────────────────
import re

DPO_TRAIN_JSONL = 'dpo_training.jsonl'
DPO_VAL_JSONL   = 'dpo_validation.jsonl'

# Re-seed so DPO generation is deterministic regardless of what consumed
# the global RNG between Section 0.5 and here.
_DPO_RNG = random.Random(SEED + 1)

_CORRUPTION_WEIGHTS = {
    'wrong_value':       0.50,
    'wrong_type':        0.25,
    'format_violation':  0.15,
    'trace_error':       0.10,
}


def _extract_answer_line(assistant_content):
    # Return (prefix_text, answer_line) where answer_line is the final
    # 'ANSWER: ...' line (without the 'ANSWER: ' prefix) and prefix_text is
    # everything before it.  If no ANSWER: line is found, treat the whole
    # string as the answer line (covers INCLUDE_REASONING=False).
    m = re.search(r'(?ms)^ANSWER:\s*(.*?)\s*$', assistant_content)
    if not m:
        return '', assistant_content.strip()
    prefix = assistant_content[:m.start()].rstrip()
    answer = m.group(1).strip()
    return prefix, answer


def _rebuild_assistant(prefix, answer_line, *, drop_prefix=False):
    if drop_prefix:
        # Format-violation: drop the ANSWER: prefix entirely.
        return (prefix + '\n\n' + answer_line).strip() if prefix else answer_line
    if prefix:
        return f'{prefix}\n\nANSWER: {answer_line}'
    return f'ANSWER: {answer_line}'


def _corrupt_unique(prefix, answer_line, gt, n_vars):
    # answer_line looks like 'x=2, y=-1, z=0'.
    var_names = VARIABLES[:n_vars]
    bad_gt = dict(gt)
    var_to_change = _DPO_RNG.choice(var_names)
    orig = bad_gt[var_to_change]
    candidates = [orig + d for d in [-3, -2, -1, 1, 2, 3]] + [-orig]
    candidates = [c for c in candidates if c != orig]
    bad_gt[var_to_change] = _DPO_RNG.choice(candidates)
    bad_answer = ', '.join(f'{v}={bad_gt[v]}' for v in var_names)
    return _rebuild_assistant(prefix, bad_answer)


def _corrupt_wrong_type(prefix, answer_line, solution_type, gt, n_vars):
    var_names = VARIABLES[:n_vars]
    if solution_type == 'unique':
        bad = _DPO_RNG.choice(['NO SOLUTION', 'INFINITE'])
        return _rebuild_assistant(prefix, bad)
    elif solution_type == 'no_solution':
        if _DPO_RNG.random() < 0.5:
            return _rebuild_assistant(prefix, 'INFINITE')
        fake = ', '.join(f'{v}={_DPO_RNG.randint(-3, 3)}' for v in var_names)
        return _rebuild_assistant(prefix, fake)
    else:  # infinite
        if _DPO_RNG.random() < 0.5:
            return _rebuild_assistant(prefix, 'NO SOLUTION')
        fake = ', '.join(f'{v}={_DPO_RNG.randint(-3, 3)}' for v in var_names)
        return _rebuild_assistant(prefix, fake)


def _corrupt_format(prefix, answer_line, solution_type, gt, n_vars):
    var_names = VARIABLES[:n_vars]
    mode = _DPO_RNG.choice(['drop_prefix', 'shuffle', 'omit'])

    if solution_type != 'unique':
        if mode == 'drop_prefix':
            return _rebuild_assistant(prefix, answer_line, drop_prefix=True)
        # Lowercase the keyword — parser is case-sensitive on these tokens.
        return _rebuild_assistant(prefix, answer_line.lower())

    if mode == 'drop_prefix':
        return _rebuild_assistant(prefix, answer_line, drop_prefix=True)
    if mode == 'shuffle':
        shuffled = list(var_names)
        _DPO_RNG.shuffle(shuffled)
        if shuffled == list(var_names) and len(var_names) > 1:
            shuffled[0], shuffled[1] = shuffled[1], shuffled[0]
        bad = ', '.join(f'{v}={gt[v]}' for v in shuffled)
        return _rebuild_assistant(prefix, bad)
    if len(var_names) <= 1:
        return _rebuild_assistant(prefix, answer_line, drop_prefix=True)
    drop_idx = _DPO_RNG.randrange(len(var_names))
    kept = [v for i, v in enumerate(var_names) if i != drop_idx]
    bad = ', '.join(f'{v}={gt[v]}' for v in kept)
    return _rebuild_assistant(prefix, bad)


def _corrupt_trace_error(prefix, answer_line, solution_type, gt, n_vars):
    # Keep the reasoning trace but flip the final ANSWER: line.  Falls back
    # to wrong_value / wrong_type when no trace exists (INCLUDE_REASONING=False).
    if solution_type == 'unique':
        return _corrupt_unique(prefix, answer_line, gt, n_vars)
    return _corrupt_wrong_type(prefix, answer_line, solution_type, gt, n_vars)


def _make_non_preferred(assistant_content, solution_type, gt, n_vars):
    prefix, answer_line = _extract_answer_line(assistant_content)
    strategies = list(_CORRUPTION_WEIGHTS.keys())
    weights    = list(_CORRUPTION_WEIGHTS.values())

    # wrong_value only applies to unique solutions; redraw if it's picked
    # for a non-unique system.
    for _attempt in range(5):
        strategy = _DPO_RNG.choices(strategies, weights=weights, k=1)[0]
        if strategy == 'wrong_value' and solution_type != 'unique':
            continue
        break
    else:
        strategy = 'wrong_type'

    if strategy == 'wrong_value':
        bad = _corrupt_unique(prefix, answer_line, gt, n_vars)
    elif strategy == 'wrong_type':
        bad = _corrupt_wrong_type(prefix, answer_line, solution_type, gt, n_vars)
    elif strategy == 'format_violation':
        bad = _corrupt_format(prefix, answer_line, solution_type, gt, n_vars)
    else:
        bad = _corrupt_trace_error(prefix, answer_line, solution_type, gt, n_vars)

    # Last-resort safety: never return an identical pair.
    if bad.strip() == assistant_content.strip():
        bad = _rebuild_assistant(
            prefix,
            'NO SOLUTION' if solution_type != 'no_solution' else 'INFINITE',
        )
    return bad, strategy


def make_dpo_record(sft_record, eval_record):
    # Convert (sft, eval) into Azure DPO preference format:
    #   { "input": {"messages": [system, user]},
    #     "preferred_output":     [{"role": "assistant", "content": <good>}],
    #     "non_preferred_output": [{"role": "assistant", "content": <bad>}] }
    sys_msg, user_msg, assistant_gold = sft_record['messages']
    preferred = assistant_gold['content']

    non_preferred, strategy = _make_non_preferred(
        preferred,
        eval_record['solution_type'],
        eval_record['ground_truth'],
        eval_record['n_vars'],
    )

    return {
        'input': {
            'messages': [
                {'role': sys_msg['role'],  'content': sys_msg['content']},
                {'role': user_msg['role'], 'content': user_msg['content']},
            ],
        },
        'preferred_output':     [{'role': 'assistant', 'content': preferred}],
        'non_preferred_output': [{'role': 'assistant', 'content': non_preferred}],
    }, strategy


def build_dpo_split(sft_records, eval_records, label):
    assert len(sft_records) == len(eval_records), \
        f'sft and eval splits out of sync: {len(sft_records)} vs {len(eval_records)}'
    dpo_records = []
    strategy_counts = Counter()
    for sft_r, eval_r in zip(sft_records, eval_records):
        rec, strat = make_dpo_record(sft_r, eval_r)
        dpo_records.append(rec)
        strategy_counts[strat] += 1
    print(f'  [{label}] {len(dpo_records)} pairs built')
    print(f'         strategy mix: {dict(strategy_counts)}')
    return dpo_records


print('Building DPO training pairs ...')
dpo_train = build_dpo_split(train_sft, train_eval, 'TRAIN')

print('\nBuilding DPO validation pairs ...')
dpo_val = build_dpo_split(val_sft, val_eval, 'VAL')

print('\nWriting DPO JSONL files ...')
write_jsonl(dpo_train, DPO_TRAIN_JSONL)
write_jsonl(dpo_val,   DPO_VAL_JSONL)

# ── Sanity check: print one example from the train split ────────────────────
print('\n──────── Sample DPO record (first from training) ────────')
rec = dpo_train[0]
print(f"USER:\n{rec['input']['messages'][1]['content']}")
print(f"\nPREFERRED:\n{rec['preferred_output'][0]['content']}")
print(f"\nNON_PREFERRED:\n{rec['non_preferred_output'][0]['content']}")


Building DPO training pairs ...
  [TRAIN] 10000 pairs built
         strategy mix: {'wrong_value': 3365, 'wrong_type': 3350, 'trace_error': 1303, 'format_violation': 1982}

Building DPO validation pairs ...
  [VAL] 1000 pairs built
         strategy mix: {'format_violation': 202, 'wrong_type': 339, 'wrong_value': 334, 'trace_error': 125}

Writing DPO JSONL files ...
  wrote 10000 records  ->  dpo_training.jsonl  (10239.2 KB)
  wrote  1000 records  ->  dpo_validation.jsonl  (1030.6 KB)

──────── Sample DPO record (first from training) ────────
USER:
3x - 5y - z = 18
-3x - 5y - 2z = 8
-4x - y + 4z = -14

PREFERRED:
3x - 5y - z = 18
-3x - 5y - 2z = 8
-4x - y + 4z = -14

R2 -> R2 + R1: 10y + 3z = -26
R3 -> 3*R3 + 4*R1: 23y - 8z = -30
R1 -> 10*R1 + 5*R2: 6x + z = 10
R3 -> 10*R3 - 23*R2: z = -2
R1 -> R1 - R3: x = 2
R2 -> R2 - 3*R3: y = -2

ANSWER: x=2, y=-2, z=-2

NON_PREFERRED:
3x - 5y - z = 18
-3x - 5y - 2z = 8
-4x - y + 4z = -14

R2 -> R2 + R1: 10y + 3z = -26
R3 -> 3*R3 + 4*R1: 23y - 8z =

## Section A — Build Three Smaller DPO Datasets

We build a single 2.5k-example pool of `(sft_record, eval_record)` pairs,
then convert it into DPO preference data **three different ways** by
swapping the corruption-strategy mix.

The shared 2.5k pool keeps Runs B/C/D identical on the prompt side — only
the `non_preferred_output` differs across them. That makes the comparison
clean.


In [9]:
# ── Generate a single shared 2.5k training pool + 250-example val pool ────
# (Smaller than the main notebook's splits; we want fast training.)
import random as _random
import numpy as _np

# Re-seed for the small splits so they're reproducible and disjoint from
# the main notebook's seed stream.
_random.seed(SEED + 100)
_np.random.seed(SEED + 100)

print('Generating extra-run training split (2500) ...')
extra_train_sft, extra_train_eval, _ = _generate_split(2500, 'EXTRA_TRAIN')

print('\nGenerating extra-run validation split (250) ...')
extra_val_sft,   extra_val_eval,   _ = _generate_split(250,  'EXTRA_VAL')

print(f'\nPool ready: {len(extra_train_sft)} train / {len(extra_val_sft)} val')


Generating extra-run training split (2500) ...
  [EXTRA_TRAIN] unique       ... 1667 done
  [EXTRA_TRAIN] no_solution  ... 417 done
  [EXTRA_TRAIN] infinite     ... 416 done

Generating extra-run validation split (250) ...
  [EXTRA_VAL] unique       ... 166 done
  [EXTRA_VAL] no_solution  ... 42 done
  [EXTRA_VAL] infinite     ... 42 done

Pool ready: 2500 train / 250 val


In [10]:
# ── Run B: default mixed corruption strategy (50/25/15/10) ─────────────────
# This matches the main notebook's _CORRUPTION_WEIGHTS exactly.
print('Building Run B preference pairs (default mix) ...')
_DPO_RNG = random.Random(SEED + 200)   # reset for reproducibility
_CORRUPTION_WEIGHTS = {
    'wrong_value':       0.50,
    'wrong_type':        0.25,
    'format_violation':  0.15,
    'trace_error':       0.10,
}
runB_train = build_dpo_split(extra_train_sft, extra_train_eval, 'B-TRAIN')
runB_val   = build_dpo_split(extra_val_sft,   extra_val_eval,   'B-VAL')

write_jsonl(runB_train, 'dpo_runB_train.jsonl')
write_jsonl(runB_val,   'dpo_runB_val.jsonl')


Building Run B preference pairs (default mix) ...
  [B-TRAIN] 2500 pairs built
         strategy mix: {'wrong_value': 886, 'trace_error': 348, 'wrong_type': 796, 'format_violation': 470}
  [B-VAL] 250 pairs built
         strategy mix: {'wrong_value': 82, 'wrong_type': 96, 'format_violation': 42, 'trace_error': 30}
  wrote  2500 records  ->  dpo_runB_train.jsonl  (2561.5 KB)
  wrote   250 records  ->  dpo_runB_val.jsonl  (252.9 KB)


In [11]:
# ── Run C: trace-error-only corruption ─────────────────────────────────────
# Every non_preferred output keeps the gold reasoning trace but flips the
# final ANSWER: line. Tests whether DPO learns most from trace-vs-answer
# inconsistency alone.
print('Building Run C preference pairs (trace-error only) ...')
_DPO_RNG = random.Random(SEED + 300)
_CORRUPTION_WEIGHTS = {
    'wrong_value':       0.0,
    'wrong_type':        0.0,
    'format_violation':  0.0,
    'trace_error':       1.0,
}
runC_train = build_dpo_split(extra_train_sft, extra_train_eval, 'C-TRAIN')
runC_val   = build_dpo_split(extra_val_sft,   extra_val_eval,   'C-VAL')

write_jsonl(runC_train, 'dpo_runC_train.jsonl')
write_jsonl(runC_val,   'dpo_runC_val.jsonl')


Building Run C preference pairs (trace-error only) ...
  [C-TRAIN] 2500 pairs built
         strategy mix: {'trace_error': 2500}
  [C-VAL] 250 pairs built
         strategy mix: {'trace_error': 250}
  wrote  2500 records  ->  dpo_runC_train.jsonl  (2564.5 KB)
  wrote   250 records  ->  dpo_runC_val.jsonl  (253.2 KB)


In [12]:
# ── Run D: default mixed corruption (same as B) ────────────────────────────
# D will use the same data as B but a different LR multiplier in the job
# config. We rebuild the pairs anyway for symmetry / a different seed.
print('Building Run D preference pairs (default mix, different seed) ...')
_DPO_RNG = random.Random(SEED + 400)
_CORRUPTION_WEIGHTS = {
    'wrong_value':       0.50,
    'wrong_type':        0.25,
    'format_violation':  0.15,
    'trace_error':       0.10,
}
runD_train = build_dpo_split(extra_train_sft, extra_train_eval, 'D-TRAIN')
runD_val   = build_dpo_split(extra_val_sft,   extra_val_eval,   'D-VAL')

write_jsonl(runD_train, 'dpo_runD_train.jsonl')
write_jsonl(runD_val,   'dpo_runD_val.jsonl')


Building Run D preference pairs (default mix, different seed) ...
  [D-TRAIN] 2500 pairs built
         strategy mix: {'wrong_value': 834, 'format_violation': 495, 'trace_error': 340, 'wrong_type': 831}
  [D-VAL] 250 pairs built
         strategy mix: {'wrong_type': 87, 'wrong_value': 80, 'format_violation': 46, 'trace_error': 37}
  wrote  2500 records  ->  dpo_runD_train.jsonl  (2561.2 KB)
  wrote   250 records  ->  dpo_runD_val.jsonl  (252.9 KB)


In [ ]:
# ── Quick spot-check: print one preferred/non-preferred pair from each run ─
import json as _json

for run, path in [('B', 'dpo_runB_train.jsonl'),
                   ('C', 'dpo_runC_train.jsonl'),
                   ('D', 'dpo_runD_train.jsonl')]:
    print(f'\n────────── Run {run}: first record ──────────')
    with open(path, 'r', encoding='utf-8-sig') as f:
        rec = _json.loads(f.readline())
    print(f"USER:\n{rec['input']['messages'][1]['content']}")
    print(f"\nNON_PREFERRED (truncated):")
    np_text = rec['non_preferred_output'][0]['content']
    print(np_text[:500] + ('...' if len(np_text) > 500 else ''))



────────── Run B: first record ──────────
USER:
-3x - 4z = -17
-6x + 6y + 4z = 2
-6y - 5z = -22

NON_PREFERRED (truncated):
-3x - 4z = -17
-6x + 6y + 4z = 2
-6y - 5z = -22

R1 -> -R1: 3x + 4z = 17
R2 -> R2 + 2*R1: y + 2z = 6
R3 -> R3 + 6*R2: z = 2
R1 -> R1 - 4*R3: x = 3
R2 -> R2 - 2*R3: y = 2

ANSWER: x=3, y=4, z=2

────────── Run C: first record ──────────
USER:
-3x - 4z = -17
-6x + 6y + 4z = 2
-6y - 5z = -22

NON_PREFERRED (truncated):
-3x - 4z = -17
-6x + 6y + 4z = 2
-6y - 5z = -22

R1 -> -R1: 3x + 4z = 17
R2 -> R2 + 2*R1: y + 2z = 6
R3 -> R3 + 6*R2: z = 2
R1 -> R1 - 4*R3: x = 3
R2 -> R2 - 2*R3: y = 2

ANSWER: x=3, y=2, z=-1

────────── Run D: first record ──────────
USER:
-3x - 4z = -17
-6x + 6y + 4z = 2
-6y - 5z = -22

NON_PREFERRED (truncated):
-3x - 4z = -17
-6x + 6y + 4z = 2
-6y - 5z = -22

R1 -> -R1: 3x + 4z = 17
R2 -> R2 + 2*R1: y + 2z = 6
R3 -> R3 + 6*R2: z = 2
R1 -> R1 - 4*R3: x = 3
R2 -> R2 - 2*R3: y = 2

ANSWER: x=3, y=4, z=2


## Section 1 — Install Azure SDKs

Skip this cell if you already have the packages from a previous notebook session in the same Colab runtime.

In [13]:
%pip install -q \
  "azure-ai-projects>=2.0.0b1" \
  openai \
  azure-identity \
  azure-mgmt-cognitiveservices \
  "azure-ai-evaluation>=1.13.0" \
  python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.3/274.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.4/220.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Section 2 — Azure Auth & OpenAI Client

Same configuration as the main notebook. Make sure `RESOURCE_GROUP` and
`OPENAI_API_KEY` are set correctly before running.

In [14]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.cognitiveservices.models import Deployment, DeploymentProperties, DeploymentModel, Sku

In [15]:
# TODO: Update RESOURCE_GROUP and OPENAI_API_KEY provided to your team via Gradescope.
# Do not modify other fields.
RESOURCE_GROUP   = 'cis-5270-team-11'
OPENAI_API_KEY   = 'TODO: INSERT AZURE OPENAI API KEY'  # ← load from .env in a real project
OPENAI_ENDPOINT  = f'https://{RESOURCE_GROUP}.openai.azure.com'
SUBSCRIPTION_ID  = '5878da59-4f37-415e-bc8c-3a1be9ba0161'  # ← load from .env in a real project


In [16]:
# Azure resource targeting
os.environ['AZURE_SUBSCRIPTION_ID'] = SUBSCRIPTION_ID
os.environ['AZURE_RESOURCE_GROUP']  = 'CIS-5270'
os.environ['AZURE_AOAI_ACCOUNT']    = RESOURCE_GROUP
os.environ['AZURE_OPENAI_API_KEY']  = OPENAI_API_KEY
os.environ['AZURE_OPENAI_ENDPOINT'] = OPENAI_ENDPOINT
CREDENTIAL = DefaultAzureCredential()

In [17]:
openai_client = AzureOpenAI(
    api_key      = OPENAI_API_KEY,
    azure_endpoint = OPENAI_ENDPOINT,
    api_version  = '2025-04-01-preview',
)
model_name = 'gpt-4.1-nano-2025-04-14'
print('Connected to Azure OpenAI')

Connected to Azure OpenAI


## Section B — Upload All Three DPO Datasets

Each run needs its own training + validation file ID. We upload all six
files now so the three jobs in Section C can launch in parallel.

In [ ]:
extra_run_files = {}

for run in ['B', 'C', 'D']:
    train_path = f'dpo_run{run}_train.jsonl'
    val_path   = f'dpo_run{run}_val.jsonl'

    print(f'Uploading Run {run} training file ...')
    with open(train_path, 'rb') as f:
        train_obj = openai_client.files.create(file=f, purpose='fine-tune')
    print(f'Uploading Run {run} validation file ...')
    with open(val_path, 'rb') as f:
        val_obj = openai_client.files.create(file=f, purpose='fine-tune')

    extra_run_files[run] = {
        'train_id': train_obj.id,
        'val_id':   val_obj.id,
    }
    print(f'  Run {run}: train={train_obj.id}  val={val_obj.id}')

print('\nWaiting for all six files to finish processing ...')
for run in ['B', 'C', 'D']:
    openai_client.files.wait_for_processing(extra_run_files[run]['train_id'])
    openai_client.files.wait_for_processing(extra_run_files[run]['val_id'])
print('All files ready!')


Uploading Run B training file ...
Uploading Run B validation file ...
  Run B: train=file-3174d39ad93c464f8e0d4cbc0e7de626  val=file-13797f0194484dbbbec5d79f95191f39
Uploading Run C training file ...
Uploading Run C validation file ...
  Run C: train=file-2c96ffe3f78a4fa0bec239b3fff372be  val=file-3697a6f8d1d8426ea6a163b7aeaf6e77
Uploading Run D training file ...
Uploading Run D validation file ...
  Run D: train=file-b8c19b9b8fbf4406a2cc5b450ad7d269  val=file-d809261e33844e178846bfb4b7b6b506

Waiting for all six files to finish processing ...
All files ready!


## Section C — Launch Three DPO Jobs in Parallel

Each job uses a different hyperparameter / dataset combination as described
in the table at the top of the notebook. All three are submitted
back-to-back so they queue together on the GlobalStandard pool.

In [ ]:
model_name = 'gpt-4.1-nano-2025-04-14'

# Per-run config: (hyperparams dict, training_file_id, validation_file_id, suffix)
extra_run_configs = {
    'B': {
        'hyperparameters': {
            'n_epochs':                 1,
            'batch_size':               4,
            'learning_rate_multiplier': 1.0,
        },
        'suffix': 'lin-eq-dpo-B',
    },
    'C': {
        'hyperparameters': {
            'n_epochs':                 1,
            'batch_size':               4,
            'learning_rate_multiplier': 1.0,
        },
        'suffix': 'lin-eq-dpo-C',
    },
    'D': {
        'hyperparameters': {
            'n_epochs':                 1,
            'batch_size':               4,
            'learning_rate_multiplier': 2.0,
        },
        'suffix': 'lin-eq-dpo-D',
    },
}

extra_run_jobs = {}

for run in ['B', 'C', 'D']:
    cfg = extra_run_configs[run]
    print(f'\nCreating Run {run} DPO job ...')
    print(f'  hyperparameters: {cfg["hyperparameters"]}')
    job = openai_client.fine_tuning.jobs.create(
        model           = model_name,
        training_file   = extra_run_files[run]['train_id'],
        validation_file = extra_run_files[run]['val_id'],
        method = {
            'type': 'dpo',
            'dpo': {'hyperparameters': cfg['hyperparameters']},
        },
        extra_body = {'trainingType': 'GlobalStandard'},
        suffix     = cfg['suffix'],
    )
    extra_run_jobs[run] = job.id
    print(f'  Job ID: {job.id}  Status: {job.status}')

print('\n─── Job ID summary ───')
for run, jid in extra_run_jobs.items():
    print(f'  Run {run}: {jid}')

# IMPORTANT: also paste these IDs into the markdown cell below in case the
# kernel dies and you lose extra_run_jobs.



Creating Run B DPO job ...
  hyperparameters: {'n_epochs': 1, 'batch_size': 4, 'learning_rate_multiplier': 1.0}
  Job ID: ftjob-22d497c032b64dbf92a8ce2cb56aff1c  Status: pending

Creating Run C DPO job ...
  hyperparameters: {'n_epochs': 1, 'batch_size': 4, 'learning_rate_multiplier': 1.0}
  Job ID: ftjob-d416cf36650e42be80b3973f8fad3551  Status: pending

Creating Run D DPO job ...
  hyperparameters: {'n_epochs': 1, 'batch_size': 4, 'learning_rate_multiplier': 2.0}
  Job ID: ftjob-f6bc5fae36384d5e803c6fdbe8aebdb3  Status: pending

─── Job ID summary ───
  Run B: ftjob-22d497c032b64dbf92a8ce2cb56aff1c
  Run C: ftjob-d416cf36650e42be80b3973f8fad3551
  Run D: ftjob-f6bc5fae36384d5e803c6fdbe8aebdb3


### Saved Job IDs

Once you've run the cell above, copy the job IDs printed and paste them here
as a backup, e.g.:

- Run B: `ftjob-22d497c032b64dbf92a8ce2cb56aff1c`
- Run C: `ftjob-d416cf36650e42be80b3973f8fad3551`
- Run D: `ftjob-f6bc5fae36384d5e803c6fdbe8aebdb3`


## Section D — Monitor All Three Jobs

Re-run the next cell periodically to check progress. Expected total
training duration: 2–4 hours per job (they run in parallel).

In [ ]:
# If you've lost extra_run_jobs (kernel died, etc.), uncomment and fill in:
# extra_run_jobs = {
#     'B': 'ftjob-xxxxxxxxxxxx',
#     'C': 'ftjob-xxxxxxxxxxxx',
#     'D': 'ftjob-xxxxxxxxxxxx',
# }
extra_run_jobs = {
    'B': 'ftjob-d416cf36650e42be80b3973f8fad3551',
    'C': 'ftjob-22d497c032b64dbf92a8ce2cb56aff1c',
    'D': 'ftjob-f6bc5fae36384d5e803c6fdbe8aebdb3',
}

print('Status check:')
for run, jid in extra_run_jobs.items():
    status = openai_client.fine_tuning.jobs.retrieve(jid).status
    print(f'  Run {run} [{jid}]: {status}')


Status check:
  Run B [ftjob-d416cf36650e42be80b3973f8fad3551]: succeeded
  Run C [ftjob-22d497c032b64dbf92a8ce2cb56aff1c]: succeeded
  Run D [ftjob-f6bc5fae36384d5e803c6fdbe8aebdb3]: succeeded


In [ ]:
# Show recent events for one specific run (change the variable to inspect others)
RUN_TO_INSPECT = 'D'   # 'B', 'C', or 'D'

events = openai_client.fine_tuning.jobs.list_events(
    extra_run_jobs[RUN_TO_INSPECT], limit=15
).data
print(f'─── Run {RUN_TO_INSPECT} recent events ───')
for ev in events:
    print(ev.message)


─── Run D recent events ───
Training tokens billed: 1296000
Completed results file: file-55cba310bc6849338eb8408d9fac8adb
Model Evaluation Passed.
Job succeeded.
Step 625: training loss=0.5187886953353882
Step 620: training loss=0.5243774652481079
Step 610: training loss=0.4555136561393738
Step 600: training loss=0.47698327898979187
Step 590: training loss=0.5394982099533081
Step 580: training loss=0.4914517104625702
Step 570: training loss=0.4855721592903137
Step 560: training loss=0.43756699562072754
Step 550: training loss=0.47340092062950134
Step 540: training loss=0.4350283145904541
Step 530: training loss=0.5550152063369751


## Section E — Deploy Each Successful Model

Wait until all three runs are `succeeded`, then run the cells below to
deploy them. Each gets its own deployment name so you can eval them
independently.

> **Quota note:** The TAs confirmed two simultaneous deployments are
> allowed on the CIS-5270 resource. You may need to deploy + eval one at a
> time, then tear down before the next, OR check whether the original Run A
> deployment is still up. Adjust the `RUNS_TO_DEPLOY` list below
> accordingly if quota is tight.


In [ ]:
# ── Pull fine_tuned_model_id for each succeeded run ────────────────────────
extra_run_models = {}
for run, jid in extra_run_jobs.items():
    job = openai_client.fine_tuning.jobs.retrieve(jid)
    if job.status == 'succeeded':
        extra_run_models[run] = job.fine_tuned_model
        print(f'  Run {run}: {job.fine_tuned_model}')
    else:
        print(f'  Run {run}: {job.status}  (skip — not finished)')


  Run B: gpt-4.1-nano-2025-04-14.ft-d416cf36650e42be80b3973f8fad3551-lin-eq-dpo-C
  Run C: gpt-4.1-nano-2025-04-14.ft-22d497c032b64dbf92a8ce2cb56aff1c-lin-eq-dpo-B
  Run D: gpt-4.1-nano-2025-04-14.ft-f6bc5fae36384d5e803c6fdbe8aebdb3-lin-eq-dpo-D


In [ ]:
# ── Deploy. Edit RUNS_TO_DEPLOY based on quota / which finished. ───────────
from azure.mgmt.cognitiveservices.models import (
    Deployment, DeploymentModel, DeploymentProperties, Sku
)

RUNS_TO_DEPLOY = ['B']   # comment some out if quota is tight

extra_run_deployments = {}

with CognitiveServicesManagementClient(credential=CREDENTIAL,
                                       subscription_id=SUBSCRIPTION_ID) as cogsvc_client:
    for run in RUNS_TO_DEPLOY:
        if run not in extra_run_models:
            print(f'Run {run} not finished — skipping deploy')
            continue

        deployment_name = f'1-nano-2025-04-14-lin-eq-dpo-{run}'
        extra_run_deployments[run] = deployment_name

        deployment_model      = DeploymentModel(format='OpenAI',
                                                name=extra_run_models[run],
                                                version='1')
        deployment_properties = DeploymentProperties(model=deployment_model)
        deployment_sku        = Sku(name='GlobalStandard', capacity=50)
        deployment_config     = Deployment(properties=deployment_properties,
                                            sku=deployment_sku)

        print(f'\nDeploying Run {run}: {deployment_name}')
        poller = cogsvc_client.deployments.begin_create_or_update(
            resource_group_name = 'CIS-5270',
            account_name        = RESOURCE_GROUP,
            deployment_name     = deployment_name,
            deployment          = deployment_config,
        )
        poller.result()
        print(f'  Deployed: {deployment_name}')

print('\n─── Deployment summary ───')
for run, name in extra_run_deployments.items():
    print(f'  Run {run}: {name}')


Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' package to be installed. You must also ensure you have the Azure Resources extension installed and have signed in to Az


Deploying Run B: 1-nano-2025-04-14-lin-eq-dpo-B


ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' package to 

In [18]:
extra_run_deployments = {
    'D': '1-nano-2025-04-14-lin-eq-dpo-D'
}

## Section F — Smoke Test & Full Evaluation

For each deployed run, we:
1. Hit the deployment with 5 random held-out problems to confirm it produces well-formed output (catches the "degenerate token loop" failure mode we saw with SFT).
2. Run the full eval harness over the held-out evaluation set.

The eval harness from the main notebook (`run_eval`, `compute_reward`) is
re-imported below so this notebook is self-contained.


In [19]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Tunables ──────────────────────────────────────────────────────────────────
MAX_WORKERS      = 2     # Concurrent API calls. Lower to 4–8 on 429s.
PRINT_EVERY      = 100    # Progress heartbeat (examples processed).
FAIL_FAST_AFTER  = 20     # Abort if the first N consecutive results are errors.


def _score_one(client, deployment_name, record):
    """Query the deployed model on one eval record; return a result dict."""
    raw, err = '', None
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model       = deployment_name,
                messages    = record['messages'],
                max_tokens  = REASONING_MAX_TOKENS if INCLUDE_REASONING else MAX_TOKENS,
                temperature = TEMPERATURE,
                timeout     = REQUEST_TIMEOUT,   # ← prevents infinite hangs
            )
            raw = resp.choices[0].message.content or ''
            err = None
            break
        except Exception as e:
            err = f'{type(e).__name__}: {e}'
            if '429' in str(e) or 'rate' in str(e).lower():
                time.sleep(2 ** attempt)   # 1s, 2s, 4s, 8s, 16s
                continue
            break   # non-retryable (auth, timeout, bad deployment) — fail fast

    n = record['n_vars']
    pred_type, pred_vals = parse_model_output(raw, n)
    reward = -1.0 if err is not None else compute_reward(
        pred_type, pred_vals, record['solution_type'], record['A'], record['b'], n
    )

    return {
        'solution_type': record['solution_type'],
        'n_vars':        n,
        'n_dependent':   record['n_dependent'],
        'pred_type':     pred_type,
        'reward':        reward,
        'raw_output':    raw,
        'expected':      record['answer'],
        'error':         err,
    }


def run_eval(client, deployment_name, eval_records, tag, resume=True):
    """
    Run `eval_records` through `deployment_name` with concurrency and checkpointing.

    Writes results to `eval_results_<tag>.jsonl` as they arrive. If the file
    already exists and resume=True, examples already scored are skipped.

    Fail-fast: if the first FAIL_FAST_AFTER consecutive results are all errors
    (e.g. wrong deployment name, auth failure, endpoint typo), raises
    RuntimeError immediately instead of burning through all 6k calls.

    Returns
    -------
    list[dict] — one result per record, in the same order as `eval_records`.
    """
    out_path = f'eval_results_{tag}.jsonl'

    # ── Resume support: figure out which indices are already done ───────────
    done = {}  # idx -> result dict
    if resume and os.path.exists(out_path):
        with open(out_path, encoding='utf-8') as f:
            for line in f:
                if not line.strip():
                    continue
                r = json.loads(line)
                done[r['_idx']] = r
        print(f'  [{tag}] resuming: {len(done)} / {len(eval_records)} already scored')

    todo = [i for i in range(len(eval_records)) if i not in done]
    if not todo:
        print(f'  [{tag}] nothing to do — all {len(eval_records)} already scored')
        return [done[i] for i in range(len(eval_records))]

    print(f'  [{tag}] scoring {len(todo)} examples with {MAX_WORKERS} workers ...')
    t0 = time.time()
    n_done = 0
    consec_errors = 0   # streak of consecutive error results

    # Append mode — checkpointing
    with open(out_path, 'a', encoding='utf-8') as fout:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(_score_one, client, deployment_name, eval_records[i]): i
                for i in todo
            }
            for fut in as_completed(futures):
                i = futures[fut]
                result = fut.result()
                result['_idx'] = i
                done[i] = result
                fout.write(json.dumps(result) + '\n')
                fout.flush()
                n_done += 1

                # ── Fail-fast guard ──────────────────────────────────────────
                # If the first FAIL_FAST_AFTER results are all errors, abort
                # instead of burning through the rest. Any success resets
                # the streak so transient 429s don't trigger it.
                if result['error'] is not None:
                    consec_errors += 1
                else:
                    consec_errors = 0

                if n_done >= FAIL_FAST_AFTER and consec_errors >= FAIL_FAST_AFTER:
                    last_err = result['error']
                    raise RuntimeError(
                        f'[{tag}] aborting: first {FAIL_FAST_AFTER} consecutive '
                        f'results were all errors. Most recent: {last_err}\n'
                        f'Check EVAL_DEPLOYMENT_NAME, API key, and endpoint.\n'
                        f'Delete {out_path} before retrying.'
                    )

                if n_done % PRINT_EVERY == 0 or n_done == len(todo):
                    rate     = n_done / (time.time() - t0 + 1e-9)
                    n_errors = sum(1 for r in done.values() if r.get('error'))
                    print(f'    {n_done:>5}/{len(todo)}  '
                          f'({rate:.1f} ex/s, {time.time() - t0:.1f}s elapsed, '
                          f'{n_errors} errors so far)')

    # Return in original order
    return [done[i] for i in range(len(eval_records))]


def summarize(results, label):
    """Print overall / per-type / per-n_vars / confusion breakdowns."""
    n = len(results)
    errors = sum(1 for r in results if r.get('error'))
    correct = sum(1 for r in results if r['reward'] > 0)

    print(f'\n{"=" * 72}')
    print(f'  EVAL REPORT — {label}   (n={n})')
    print(f'{"=" * 72}')
    print(f'Overall accuracy : {correct}/{n} = {correct/n:.3f}')
    if errors:
        print(f'API errors       : {errors}  (counted as incorrect)')

    # ── By solution type ─────────────────────────────────────────────────────
    print(f'\nBy solution type:')
    print(f'  {"type":<14} {"n":>6} {"correct":>9} {"acc":>7}')
    for t in ['unique', 'no_solution', 'infinite']:
        sub = [r for r in results if r['solution_type'] == t]
        if not sub:
            continue
        c = sum(1 for r in sub if r['reward'] > 0)
        print(f'  {t:<14} {len(sub):>6} {c:>9} {c/len(sub):>7.3f}')

    # ── By n_vars ────────────────────────────────────────────────────────────
    print(f'\nBy n_vars:')
    print(f'  {"n_vars":<8} {"n":>6} {"correct":>9} {"acc":>7}')
    for n_v in sorted({r['n_vars'] for r in results}):
        sub = [r for r in results if r['n_vars'] == n_v]
        c = sum(1 for r in sub if r['reward'] > 0)
        print(f'  {n_v:<8} {len(sub):>6} {c:>9} {c/len(sub):>7.3f}')

    # ── Confusion matrix: true_type × pred_type ──────────────────────────────
    types = ['unique', 'no_solution', 'infinite', 'unparseable']
    cm = {t: Counter() for t in ['unique', 'no_solution', 'infinite']}
    for r in results:
        cm[r['solution_type']][r['pred_type']] += 1

    print(f'\nConfusion matrix (rows = true, cols = predicted):')
    header = f'  {"true \\ pred":<14}' + ''.join(f'{t:>13}' for t in types) + f'{"total":>8}'
    print(header)
    for t in ['unique', 'no_solution', 'infinite']:
        row_total = sum(cm[t].values())
        row = f'  {t:<14}' + ''.join(f'{cm[t][p]:>13}' for p in types) + f'{row_total:>8}'
        print(row)

    # ── By n_vars × solution_type (hardest-cell view) ────────────────────────
    print(f'\nAccuracy by (n_vars, solution_type):')
    print(f'  {"n_vars":<8}' + ''.join(f'{t:>14}' for t in ['unique', 'no_solution', 'infinite']))
    for n_v in sorted({r['n_vars'] for r in results}):
        row = f'  {n_v:<8}'
        for t in ['unique', 'no_solution', 'infinite']:
            sub = [r for r in results if r['n_vars'] == n_v and r['solution_type'] == t]
            if sub:
                c = sum(1 for r in sub if r['reward'] > 0)
                row += f'{c/len(sub):>13.3f} '
            else:
                row += f'{"—":>13} '
        print(row)

    # Return structured aggregates for downstream comparison
    return {
        'label':    label,
        'n':        n,
        'accuracy': correct / n,
        'errors':   errors,
        'by_type':  {t: {
            'n': sum(1 for r in results if r['solution_type'] == t),
            'correct': sum(1 for r in results if r['solution_type'] == t and r['reward'] > 0),
        } for t in ['unique', 'no_solution', 'infinite']},
        'by_n_vars': {n_v: {
            'n': sum(1 for r in results if r['n_vars'] == n_v),
            'correct': sum(1 for r in results if r['n_vars'] == n_v and r['reward'] > 0),
        } for n_v in sorted({r['n_vars'] for r in results})},
    }


def assert_eval_healthy(results, tag, max_error_rate=0.05):
    """
    Raise if more than max_error_rate of results are API errors.

    Call this right after run_eval() + summarize(). Prevents silently
    recording a garbage eval (e.g., the 0/6000 case where every call
    hit APIConnectionError and was scored as -1).
    """
    n = len(results)
    err_count = sum(1 for r in results if r.get('error'))
    err_rate = err_count / n if n else 0.0
    print(f'[{tag}] health check: {err_count}/{n} errors ({err_rate:.1%})')
    if err_rate > max_error_rate:
        top = Counter(r['error'] for r in results if r.get('error')).most_common(3)
        raise RuntimeError(
            f'[{tag}] eval is unhealthy: {err_rate:.1%} error rate '
            f'(>{max_error_rate:.0%} threshold).\n'
            f'Top errors: {top}\n'
            f'Delete eval_results_{tag}.jsonl and fix the root cause before re-running.'
        )
    print(f'[{tag}] healthy — proceed')



def fetch_training_metadata(client, fine_tuned_model_name):
    """
    Search Azure fine-tuning jobs for one whose fine_tuned_model field matches
    fine_tuned_model_name.  Returns a metadata dict, or None for baseline models.

    Available fields: job_id, base_model, method (supervised/dpo/reinforcement),
    hyperparameters, training_file, validation_file, trained_tokens, timestamps.
    """
    try:
        for job in client.fine_tuning.jobs.list():
            if job.fine_tuned_model == fine_tuned_model_name:
                hp = job.hyperparameters
                return {
                    'job_id':           job.id,
                    'base_model':       job.model,
                    'method':           str(job.method),
                    'hyperparameters': {
                        'n_epochs':                 hp.n_epochs,
                        'batch_size':               hp.batch_size,
                        'learning_rate_multiplier': hp.learning_rate_multiplier,
                    },
                    'training_file':    job.training_file,
                    'validation_file':  job.validation_file,
                    'trained_tokens':   job.trained_tokens,
                    'created_at':       job.created_at,
                    'finished_at':      job.finished_at,
                    'seed':             getattr(job, 'seed', None),
                    'status':           job.status,
                }
    except Exception as e:
        print(f'  [fetch_training_metadata] warning: {e}')
    return None


print('Eval harness loaded (v2, fail-fast).')
print('  run_eval(client, deployment_name, eval_records, tag, resume=True)  ->  list[result]')
print('     └─ aborts if first', FAIL_FAST_AFTER, 'consecutive results are all errors')
print('  summarize(results, label)                                          ->  dict of aggregates')
print('  assert_eval_healthy(results, tag, max_error_rate=0.05)             ->  raises if unhealthy')
print('  fetch_training_metadata(client, model_name)                          ->  metadata dict or None')


Eval harness loaded (v2, fail-fast).
  run_eval(client, deployment_name, eval_records, tag, resume=True)  ->  list[result]
     └─ aborts if first 20 consecutive results are all errors
  summarize(results, label)                                          ->  dict of aggregates
  assert_eval_healthy(results, tag, max_error_rate=0.05)             ->  raises if unhealthy
  fetch_training_metadata(client, model_name)                          ->  metadata dict or None


In [20]:
# ── Generate a fresh held-out eval set for these runs ─────────────────────
# (Same seed as the main notebook so results are comparable.)
import random as _random, numpy as _np
_random.seed(SEED)
_np.random.seed(SEED)

# Burn the same number of random draws as the main notebook so the
# extra-run eval set is the SAME held-out set used in the report.
print('Regenerating training/val/eval splits to recover held-out eval set ...')
_t_sft, _t_ev, _ = _generate_split(N_TRAIN_TOTAL, 'TRAIN-burn')
_v_sft, _v_ev, _ = _generate_split(N_VAL_TOTAL,   'VAL-burn')
_, held_out_eval, _ = _generate_split(N_EVAL_TOTAL, 'EVAL')
print(f'\nHeld-out eval set: {len(held_out_eval)} examples')


Regenerating training/val/eval splits to recover held-out eval set ...
  [TRAIN-burn] unique       ... 6667 done
  [TRAIN-burn] no_solution  ... 1667 done
  [TRAIN-burn] infinite     ... 1666 done
  [VAL-burn] unique       ... 666 done
  [VAL-burn] no_solution  ... 167 done
  [VAL-burn] infinite     ... 167 done
  [EVAL] unique       ... 666 done
  [EVAL] no_solution  ... 167 done
  [EVAL] infinite     ... 167 done

Held-out eval set: 1000 examples


In [21]:
# ── Smoke test: 5 random examples per deployment ──────────────────────────
import random as _random

for run, deployment_name in extra_run_deployments.items():
    print(f'\n========== Smoke test: Run {run} ({deployment_name}) ==========')
    samples = _random.sample(held_out_eval, 5)
    for ex in samples:
        try:
            response = openai_client.chat.completions.create(
                model       = deployment_name,
                messages    = ex['messages'],
                max_tokens  = REASONING_MAX_TOKENS,
                temperature = 0,
                timeout     = REQUEST_TIMEOUT,
            )
            raw = response.choices[0].message.content
            pred_type, pred_vals = parse_model_output(raw, ex['n_vars'])
            reward = compute_reward(pred_type, pred_vals,
                                    ex['solution_type'], ex['A'], ex['b'],
                                    ex['n_vars'])
            print(f'  true={ex["solution_type"]:<11} pred={pred_type}  reward={reward:+.1f}')
            # Print first 100 chars to spot degenerate-loop pathologies
            print(f'    raw (first 120 chars): {raw[:120]!r}')
        except Exception as e:
            print(f'  ERROR: {e}')



========== Smoke test: Run D (1-nano-2025-04-14-lin-eq-dpo-D) ==========
  true=unique      pred=unparseable  reward=-1.0
    raw (first 120 chars): 'ANSWER: x ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( '
  true=infinite    pred=unparseable  reward=-1.0
    raw (first 120 chars): 'ANSWER: x x ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( '
  true=no_solution pred=unparseable  reward=-1.0
    raw (first 120 chars): 'ANSWER: x ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( '
  true=unique      pred=unparseable  reward=-1.0
    raw (first 120 chars): 'ANSWER: x ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( '
  true=unique      pred=unparseable  reward=-1.0
    raw (first 120 chars): 'ANSWER: x ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( ( (

In [ ]:
# # ── Full eval against held-out set, one run at a time ─────────────────────
# # This re-uses run_eval / save_eval_results from the eval harness above.
# # Results are saved to eval_results_dpo_<run>.jsonl + eval_summary_dpo_<run>.json.

# import json as _json

# extra_run_eval_results = {}

# for run, deployment_name in extra_run_deployments.items():
#     print(f'\n========== Full eval: Run {run} ==========')
#     eval_tag = f'dpo_{run}'

#     # run_eval comes from the harness cell above; signature follows the main notebook.
#     results = run_eval(
#         deployment_name = deployment_name,
#         eval_set        = held_out_eval,
#         max_workers     = 2,           # match main notebook
#         fail_fast_after = 50,
#         max_tokens      = REASONING_MAX_TOKENS,
#         temperature     = 0,
#     )

#     # Save
#     out_path = f'eval_results_dpo_{run}.jsonl'
#     with open(out_path, 'w', encoding='utf-8') as f:
#         for r in results:
#             f.write(_json.dumps(r) + '\n')

#     # Summary
#     correct = sum(1 for r in results if r.get('reward', 0) > 0)
#     total   = len(results)
#     summary = {
#         'run':              run,
#         'deployment_name':  deployment_name,
#         'n_total':          total,
#         'n_correct':        correct,
#         'accuracy':         correct / total if total else 0,
#     }
#     with open(f'eval_summary_dpo_{run}.json', 'w') as f:
#         _json.dump(summary, f, indent=2)
#     extra_run_eval_results[run] = summary

#     print(f'  Run {run}: {correct}/{total} = {correct/total:.1%}')

# print('\n─── Final summary ───')
# for run, s in extra_run_eval_results.items():
#     print(f'  Run {run}: {s["accuracy"]:.1%}  ({s["n_correct"]}/{s["n_total"]})')



========== Full eval: Run B ==========


TypeError: run_eval() got an unexpected keyword argument 'eval_set'

In [22]:
extra_run_eval_results = {}

for run, deployment_name in extra_run_deployments.items():
    print(f'\n========== Full eval: Run {run} ==========')
    tag = f'dpo_{run}'
    results = run_eval(
        client          = openai_client,
        deployment_name = deployment_name,
        eval_records    = held_out_eval,
        tag             = tag,
    )
    summary = summarize(results, label=f'DPO Run {run}')
    assert_eval_healthy(results, tag)
    extra_run_eval_results[run] = summary

    with open(f'eval_summary_dpo_{run}.json', 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'Saved eval_summary_dpo_{run}.json')


========== Full eval: Run D ==========
  [dpo_D] scoring 1000 examples with 2 workers ...
      100/1000  (0.8 ex/s, 120.8s elapsed, 0 errors so far)
      200/1000  (0.8 ex/s, 239.3s elapsed, 0 errors so far)
      300/1000  (0.8 ex/s, 356.1s elapsed, 0 errors so far)
      400/1000  (0.8 ex/s, 473.1s elapsed, 0 errors so far)
      500/1000  (0.8 ex/s, 590.8s elapsed, 0 errors so far)
      600/1000  (0.9 ex/s, 704.3s elapsed, 0 errors so far)
      700/1000  (0.9 ex/s, 819.3s elapsed, 0 errors so far)
      800/1000  (0.9 ex/s, 935.2s elapsed, 0 errors so far)
      900/1000  (0.9 ex/s, 1054.1s elapsed, 0 errors so far)
     1000/1000  (0.9 ex/s, 1170.4s elapsed, 0 errors so far)

  EVAL REPORT — DPO Run D   (n=1000)
Overall accuracy : 4/1000 = 0.004

By solution type:
  type                n   correct     acc
  unique            666         0   0.000
  no_solution       167         4   0.024
  infinite          167         0   0.000

By n_vars:
  n_vars        n   correct     acc


In [ ]:
from google.colab import files
files.download('eval_summary_dpo_D.json')
files.download('eval_results_dpo_D.jsonl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Section G — (Optional) Tear Down Deployments

Once you've collected the eval results, tear down the deployments to free
up quota and avoid hourly costs. Skip this if you still need them for the
report or further experimentation.

In [ ]:
# RUNS_TO_TEARDOWN = ['B', 'C']   # e.g. ['B', 'C', 'D'] — leave empty by default

# with CognitiveServicesManagementClient(credential=CREDENTIAL,
#                                        subscription_id=SUBSCRIPTION_ID) as cogsvc_client:
#     for run in RUNS_TO_TEARDOWN:
#         deployment_name = extra_run_deployments.get(run)
#         if not deployment_name:
#             continue
#         print(f'Deleting {deployment_name} ...')
#         poller = cogsvc_client.deployments.begin_delete(
#             resource_group_name = 'CIS-5270',
#             account_name        = RESOURCE_GROUP,
#             deployment_name     = deployment_name,
#         )
#         poller.result()
#         print(f'  Deleted: {deployment_name}')


Deleting 1-nano-2025-04-14-lin-eq-dpo-B ...


Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' package to be installed. You must also ensure you have the Azure Resources extension installed and have signed in to Az

ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' package to 